# Combining cleaned CSV files - Only Matching columns

In [104]:
#Import libraries

import pandas as pd 
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np  
import sys

##### Pulling only common columns from both dataset

In [105]:
teen_df = pd.read_csv('../data/teen_screentime_addiction_cleaned.csv')
global_df = pd.read_csv('../data/global_smartphone_addiction_cleaned.csv')

In [106]:
#calling function 3

sys.path.append("..")
from functions import combined_datasets

concat_df = combined_datasets(teen_df, global_df)


Concatenated successful — no duplicate User_ID values.


In [107]:
column_order = ['User_ID', 'Source_Dataset', 'Age', 'Gender', 
                 'Daily_Usage_Hours', 'Sleep_Hours', 'Exercise_Hours',
                 'Time_on_Social_Media', 'Time_on_Gaming', 'Phone_Checks_Per_Day',
                 'Anxiety_Level', 'Depression_Level', 'Addiction_Level']

concat_df = concat_df[column_order]
concat_df.head()

,User_ID,Source_Dataset,Age,Gender,Daily_Usage_Hours,Sleep_Hours,Exercise_Hours,Time_on_Social_Media,Time_on_Gaming,Phone_Checks_Per_Day,Anxiety_Level,Depression_Level,Addiction_Level
0,1,teen_screentime,13,Female,4.0,6.1,0.1,3.6,1.7,86,10.0,3.0,10.0
1,2,teen_screentime,17,Female,5.5,6.5,0.0,1.1,4.0,96,3.0,7.0,10.0
2,3,teen_screentime,13,Other,5.8,5.5,0.8,0.3,1.5,137,2.0,3.0,9.2
3,4,teen_screentime,18,Female,3.1,3.9,1.6,3.1,1.6,128,9.0,10.0,9.8
4,5,teen_screentime,14,Other,2.5,6.7,1.1,2.6,0.9,96,1.0,5.0,8.6


##### Quick EDA check

Run .info() 

    - Confirming the total number of rows and columns
    - Confirming if there is any null values
    - Confirming datatype of all columns


In [108]:
# To make sure the combined dataset has expected details
 
concat_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   User_ID               6000 non-null   int64  
 1   Source_Dataset        6000 non-null   str    
 2   Age                   6000 non-null   int64  
 3   Gender                6000 non-null   str    
 4   Daily_Usage_Hours     6000 non-null   float64
 5   Sleep_Hours           6000 non-null   float64
 6   Exercise_Hours        6000 non-null   float64
 7   Time_on_Social_Media  6000 non-null   float64
 8   Time_on_Gaming        6000 non-null   float64
 9   Phone_Checks_Per_Day  6000 non-null   int64  
 10  Anxiety_Level         6000 non-null   float64
 11  Depression_Level      6000 non-null   float64
 12  Addiction_Level       6000 non-null   float64
dtypes: float64(8), int64(3), str(2)
memory usage: 609.5 KB


In [109]:
#  To confirm the exact column names
concat_df.columns

Index(['User_ID', 'Source_Dataset', 'Age', 'Gender', 'Daily_Usage_Hours',
       'Sleep_Hours', 'Exercise_Hours', 'Time_on_Social_Media',
       'Time_on_Gaming', 'Phone_Checks_Per_Day', 'Anxiety_Level',
       'Depression_Level', 'Addiction_Level'],
      dtype='str')

##### Build smartphone_analysis Database
**Smartphone database — 3 tables**

- `User_ID` is the primary key in all three tables, and also a foreign key in the `Duration` and `Health` tables.

    1. `Users` — General information about the user (e.g. Age, Gender) <br>
    2. `Habits` — Time spent on each activity (e.g. Daily_Usage_Hours, Sleep_Hours, Exercise_Hours, Time_on_Social_Media, Time_on_Gaming, Phone_Checks_Per_Day) <br>
    3. `Health` — Health impact (e.g. Anxiety_Level, Depression_Level, Addiction_Level)


- The `Source_Dataset` column in the `User` table identifies which original dataset each row came from.
- Inputting the actual data into appropriate tables
- Keeping all the codes in one cell so that it doesnt create any duplicate rows
(If you keep the `to_sql` code in a separate cell, and someone accidentally runs that cell alone, it will create duplicate rows — since we're using `if_exists='append'`.)


In [110]:
connection = sqlite3.connect("../Data/smartphone_analysis.db")
cursor = connection.cursor()

# Drop old tables if they exist
cursor.execute("DROP TABLE IF EXISTS Users")
cursor.execute("DROP TABLE IF EXISTS Habits")
cursor.execute("DROP TABLE IF EXISTS Health")

# Create Users table
cursor.execute("""
CREATE TABLE Users
        (
        User_ID INTEGER PRIMARY KEY,
        Source_Dataset TEXT,
        Age INTEGER,
        Gender TEXT
        )
        """)

# Create Habits table
cursor.execute("""
CREATE TABLE Habits
        (
        User_ID INTEGER PRIMARY KEY,
        Daily_Usage_Hours REAL,
        Sleep_Hours REAL,
        Exercise_Hours REAL,
        Time_on_Social_Media REAL,
        Time_on_Gaming REAL,
        Phone_Checks_Per_Day INTEGER,
        FOREIGN KEY (User_ID) REFERENCES User(User_ID)
        )
        """)

# Create Health Table
cursor.execute( """
CREATE TABLE Health
        (
        User_ID INTEGER PRIMARY KEY,
        Anxiety_Level REAL,
        Depression_Level REAL,
        Addiction_Level REAL,
        FOREIGN KEY (User_ID) REFERENCES User(User_ID)
        )
""")

connection.commit()

# Input the actual data into the tables 

users_data       = concat_df[['User_ID', 'Source_Dataset', 'Age', 'Gender']]
habits_data   = concat_df[['User_ID', 'Daily_Usage_Hours', 'Sleep_Hours', 'Exercise_Hours', 'Time_on_Social_Media', 'Time_on_Gaming', 'Phone_Checks_Per_Day']]
health_data     = concat_df[['User_ID', 'Anxiety_Level', 'Depression_Level', 'Addiction_Level']]

users_data.to_sql('Users', connection, if_exists='append', index=False)
habits_data.to_sql('Habits', connection, if_exists='append', index=False)
health_data.to_sql('Health', connection, if_exists='append', index=False)

connection.commit()


**To double check that each table has 6000 rows (from combining the teen_screentime_addiction and global_smartphone_addiction datasets)**

In [111]:
# Confirming 6000 rows in each tables

for table in ['Users', 'Habits', 'Health']:
    count = pd.read_sql(f"SELECT COUNT(*) as row_count FROM {table}", connection)
    print(table, count['row_count'][0])

Users 6000
Habits 6000
Health 6000


In [112]:
test = """  select * from Users U
            join Habits Ha on U.User_ID = Ha.User_ID
            join Health H on Ha.User_ID = H.User_ID 
            limit 5 
        """
result = pd.read_sql(test, connection)
result

,User_ID,Source_Dataset,Age,Gender,User_ID,Daily_Usage_Hours,Sleep_Hours,Exercise_Hours,Time_on_Social_Media,Time_on_Gaming,Phone_Checks_Per_Day,User_ID,Anxiety_Level,Depression_Level,Addiction_Level
0,1,teen_screentime,13,Female,1,4.0,6.1,0.1,3.6,1.7,86,1,10.0,3.0,10.0
1,2,teen_screentime,17,Female,2,5.5,6.5,0.0,1.1,4.0,96,2,3.0,7.0,10.0
2,3,teen_screentime,13,Other,3,5.8,5.5,0.8,0.3,1.5,137,3,2.0,3.0,9.2
3,4,teen_screentime,18,Female,4,3.1,3.9,1.6,3.1,1.6,128,4,9.0,10.0,9.8
4,5,teen_screentime,14,Other,5,2.5,6.7,1.1,2.6,0.9,96,5,1.0,5.0,8.6


#### SQL Queries

1. Does daily smartphone usage predict addiction level?
2. Does addiction level correlate with mental health?
3. How do usage patterns differ between the teen dataset and the broader population dataset?
4. Which age group have a daily usage hours above the overall average?
5. Does daily smartphone usage differ by gender?

##### 1. Does daily smartphone usage predict addiction level?

Daily smartphone usage shows a strong positive relationship with addiction levels, particularly between 0 to 6 hours of usage. Addiction levels shows a strong postive trend from 0-5 hours and peak around 5–7 hours per day (max 7.83). Beyond this point, addiction levels plateau, indicating diminishing impact of additional usage. This suggests that moderate-to-high usage is a key predictor of smartphone addiction.

In [113]:
# 1. Does daily smartphone usage predict addiction level?

query1 = """

    SELECT 
        ROUND(Ha.daily_usage_hours, 0) AS usage_hours,
        ROUND(AVG(H.addiction_level), 2) AS avg_addiction,
        COUNT(*) as users_count
    FROM "Users" U
    JOIN Habits Ha ON U.User_ID = Ha.User_ID
    JOIN Health H ON Ha.User_ID = H.User_ID
    GROUP BY ROUND(Ha.daily_usage_hours, 0)
    ORDER BY usage_hours DESC;

"""
result1 = pd.read_sql(query1, connection)
result1

,usage_hours,avg_addiction,users_count
0,13.0,7.50,1
1,12.0,7.36,11
2,11.0,7.49,34
3,10.0,6.73,106
4,9.0,7.20,317
5,8.0,7.55,544
6,7.0,7.81,881
7,6.0,7.83,1105
8,5.0,7.83,1121
9,4.0,7.59,934


##### 2. Does addiction correlates to mental health?

In [114]:
# 2. Does addiction correlates to mental health

query2 = """ 

SELECT 
    ROUND(Addiction_Level, 0) AS addiction_level,
    ROUND(AVG(Anxiety_Level), 1) AS avg_anxiety,
    ROUND(AVG(Depression_Level), 1) AS avg_depression,
    COUNT(*) as users_count
FROM Health
GROUP BY ROUND(Addiction_Level, 0)
ORDER BY addiction_level DESC;

"""

result2 = pd.read_sql(query2, connection)
result2

,addiction_level,avg_anxiety,avg_depression,users_count
0,10.0,5.5,5.3,2479
1,9.0,5.5,5.6,376
2,8.0,5.2,5.2,1109
3,7.0,5.4,5.0,239
4,6.0,5.1,5.1,895
5,5.0,5.9,5.3,84
6,4.0,5.3,5.1,39
7,3.0,4.9,5.1,770
8,2.0,5.8,6.5,6
9,1.0,5.0,7.7,3


##### 3. How do usage patterns differ between the teen dataset and the broader population dataset?

In [115]:
#  3. How do usage patterns differ between the teen dataset and the broader population dataset?


query3 = """

    SELECT 
        U.Source_Dataset,
        ROUND(AVG(Ha.Daily_Usage_Hours), 1) AS avg_daily_usage,
        ROUND(AVG(Ha.Sleep_Hours), 1) AS avg_sleep,
        ROUND(AVG(Ha.Exercise_Hours), 1) AS avg_exercise,
        ROUND(AVG(Ha.Time_on_Social_Media), 1) AS avg_social_media,
        ROUND(AVG(Ha.Time_on_Gaming), 1) AS avg_gaming,
        ROUND(AVG(Ha.Phone_Checks_Per_Day), 0) AS avg_phone_checks,
        COUNT(*) as users_count
    FROM Users U
    JOIN Habits Ha ON U.User_ID = Ha.User_ID
    GROUP BY U.Source_Dataset;

"""
result3 = pd.read_sql(query3, connection)
result3

,Source_Dataset,avg_daily_usage,avg_sleep,avg_exercise,avg_social_media,avg_gaming,avg_phone_checks,users_count
0,global_mobile,6.0,6.5,1.0,2.0,1.5,80.0,3000
1,teen_screentime,5.0,6.5,1.0,2.5,1.5,83.0,3000


##### 4. Which age group has daily usage hours above the overall average?

In [116]:
query4 = """

    SELECT 
        CASE 
            WHEN U.Age <= 19 THEN 'Teen and Preteens'
            WHEN U.Age BETWEEN 20 AND 29 THEN 'Twenties'
            WHEN U.Age BETWEEN 30 AND 39 THEN 'Thirties'
            WHEN U.Age BETWEEN 40 AND 49 THEN 'Fourties'
            WHEN U.Age BETWEEN 50 AND 59 THEN 'Fifties'
            ELSE '60+'
        END AS age_group,
        ROUND(AVG(Ha.Daily_Usage_Hours), 2) AS avg_daily_usage,
        COUNT(*) as users_count
    FROM Users U
    JOIN Habits Ha ON U.User_ID = Ha.User_ID
    GROUP BY age_group
    HAVING avg_daily_usage > (SELECT AVG(Daily_Usage_Hours) FROM Habits)
    ORDER BY avg_daily_usage DESC;

"""
result4 = pd.read_sql(query4, connection)
result4

,age_group,avg_daily_usage,users_count
0,Thirties,6.10,509
1,Fourties,6.04,518
2,60+,5.99,592
3,Twenties,5.96,497
4,Fifties,5.88,481


##### 5. Does smartphone daily usage differ by Gender?

In [118]:
query5 = """

    SELECT 
        U.Gender,
        ROUND(AVG(Ha.Daily_Usage_Hours), 2) AS avg_daily_usage,
        ROUND(MAX(Ha.Daily_Usage_Hours), 2) AS max_usage,
        COUNT(*) as users_count
    FROM Users U
    JOIN Habits Ha ON U.User_ID = Ha.User_ID
    GROUP BY U.Gender
    ORDER BY avg_daily_usage DESC;

"""
result5 = pd.read_sql(query5, connection)
result5

,Gender,avg_daily_usage,max_usage,users_count
0,Male,5.53,12.37,2063
1,Female,5.50,12.06,1970
2,Other,5.48,12.87,1967
